# High-z Example 04: All 8 Tier-1 Rotator Gallery

**EPS Research — High-z Kinematic Corpus Z1**

All 8 confirmed ALPINE rotators plotted in a single figure.
Each panel shows Vrot with 3DBarolo error bars, loaded directly
from the Z1 corpus JSON.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.21834678  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    if not os.path.exists("high_z_kinematic_corpus_Z1_flat.csv"):
        import json, csv
        _g = json.load(open("high_z_kinematic_corpus_Z1.json"))["galaxies"]
        _k = [k for k in _g[0] if not isinstance(_g[0][k], (dict, list))]
        with open("high_z_kinematic_corpus_Z1_flat.csv", "w", newline="") as _f:
            _w = csv.DictWriter(_f, fieldnames=_k)
            _w.writeheader()
            for _r in _g:
                _w.writerow({k: ("" if _r.get(k) is None else _r.get(k)) for k in _k})
        print("  built high_z_kinematic_corpus_Z1_flat.csv from the JSON (the Zenodo record has no flat CSV)")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('high_z_kinematic_corpus_Z1.json') as f:
    corpus = json.load(f)

galaxies  = corpus['galaxies']
rotators  = [g for g in galaxies if g.get('is_rotator') and g.get('quality_tier')==1]
print(f"Total galaxies: {len(galaxies)}")
print(f"Tier-1 rotators: {len(rotators)}")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i, g in enumerate(sorted(rotators, key=lambda x: x['redshift'])):
    d    = g['data']
    R    = [p['R_kpc']     for p in d]
    Vrot = [p['Vrot_kms']  for p in d]
    eV   = [p['e_Vrot_kms'] for p in d]
    ax   = axes[i]
    ax.errorbar(R, Vrot, yerr=eV, fmt='o-', color='#e74c3c',
                capsize=4, linewidth=1.5, markersize=5)
    ax.set_title(f"{g['galaxy']}\nz={g['redshift']:.4f}  "
                 f"Vmax={max(Vrot):.0f} km/s", fontsize=8)
    ax.set_xlabel('R (kpc)', fontsize=7)
    ax.set_ylabel('Vrot (km/s)', fontsize=7)
    ax.tick_params(labelsize=7)

plt.suptitle('All 8 ALPINE Tier-1 Rotators — EPS Research Z1 Corpus\n'
             'Sorted by redshift (z = 4.26 to 5.54)', fontsize=11)
plt.tight_layout()
plt.savefig('hz04_rotator_gallery.png', dpi=150, bbox_inches='tight')
plt.show()